In [ ]:
!pip -q install -U \
    transformers \
    datasets \
    accelerate \
    groq \
    pandas \
    tqdm \
    rouge-score \
    sentencepiece \
    psutil

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 6.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 89.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 14.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.5 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you

In [ ]:
import os
import gc
import json
import time
import random
import psutil

import torch
import pandas as pd

from tqdm.auto import tqdm
from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

In [ ]:
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print(
    "System RAM:",
    round(psutil.virtual_memory().total / 1024**3, 2),
    "GB"
)

Torch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
System RAM: 12.67 GB


In [ ]:
cnn = load_dataset(
    "abisee/cnn_dailymail",
    "3.0.0"
)

cnn_df = pd.concat(
    [
        cnn["train"].to_pandas(),
        cnn["validation"].to_pandas(),
        cnn["test"].to_pandas()
    ],
    ignore_index=True
)

cnn_df = cnn_df[["article"]].rename(
    columns={"article": "text"}
)

cnn_df["source"] = "cnn_dailymail"

print(len(cnn_df))
cnn_df.head()

README.md:   0%|          | 0.00/15.6k [00:00<?, ?B/s]

3.0.0/train-00000-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  257MB            

3.0.0/train-00000-of-00003.parquet: downloading bytes:           |  0.00B            

3.0.0/train-00001-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  257MB            

3.0.0/train-00001-of-00003.parquet: downloading bytes:           |  0.00B            

...

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

311971


,text,source
0,"LONDON, England (Reuters) -- Harry Potter star...",cnn_dailymail
1,Editor's note: In our Behind the Scenes series...,cnn_dailymail
2,"MINNEAPOLIS, Minnesota (CNN) -- Drivers who we...",cnn_dailymail
3,WASHINGTON (CNN) -- Doctors removed five small...,cnn_dailymail
4,(CNN) -- The National Football League has ind...,cnn_dailymail


In [ ]:
xsum = load_dataset(
    "EdinburghNLP/xsum"
)

xsum_df = pd.concat(
    [
        xsum["train"].to_pandas(),
        xsum["validation"].to_pandas(),
        xsum["test"].to_pandas()
    ],
    ignore_index=True
)

xsum_df = xsum_df[["document"]].rename(
    columns={"document": "text"}
)

xsum_df["source"] = "xsum"

print(len(xsum_df))
xsum_df.head()

README.md:   0%|          | 0.00/6.42k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  300MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 16.4MB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

Generating train split:   0%|          | 0/204045 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11332 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11334 [00:00<?, ? examples/s]

226711


,text,source
0,"The full cost of damage in Newton Stewart, one...",xsum
1,A fire alarm went off at the Holiday Inn in Ho...,xsum
2,Ferrari appeared in a position to challenge un...,xsum
3,"John Edward Bates, formerly of Spalding, Linco...",xsum
4,Patients and staff were evacuated from Cerahpa...,xsum


In [ ]:
ag = load_dataset(
    "fancyzhx/ag_news"
)

ag_df = pd.concat(
    [
        ag["train"].to_pandas(),
        ag["test"].to_pandas()
    ],
    ignore_index=True
)

ag_df = ag_df[["text"]]

ag_df["source"] = "ag_news"

print(len(ag_df))
ag_df.head()

README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 18.6MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.23MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

127600


,text,source
0,Wall St. Bears Claw Back Into the Black (Reute...,ag_news
1,Carlyle Looks Toward Commercial Aerospace (Reu...,ag_news
2,Oil and Economy Cloud Stocks' Outlook (Reuters...,ag_news
3,Iraq Halts Oil Exports from Main Southern Pipe...,ag_news
4,"Oil prices soar to all-time record, posing new...",ag_news


In [ ]:
aeslc = load_dataset(
    "Yale-LILY/aeslc"
)

aeslc_df = pd.concat(
    [
        aeslc["train"].to_pandas(),
        aeslc["validation"].to_pandas(),
        aeslc["test"].to_pandas()
    ],
    ignore_index=True
)

print(aeslc_df.columns)
aeslc_df.head()

README.md:   0%|          | 0.00/6.34k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 6.26MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  896kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

Generating train split:   0%|          | 0/14436 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1960 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1906 [00:00<?, ? examples/s]

Index(['email_body', 'subject_line'], dtype='str')


,email_body,subject_line
0,"Greg/Phillip, Attached is the Grande Communic...",Service Agreement
1,Phillip & Keith Attached is the first draw re...,Bishops Corner
2,Your Internet Banking accounts are now setup a...,Internet Banking
3,To our IBS Customers that are still hanging in...,Internet Banking
4,Phillip Good Morning!\nI hope you had a wonder...,SMEs for expert stories


In [ ]:
aeslc_df = aeslc_df[
    ["email_body"]
].rename(
    columns={"email_body": "text"}
)

aeslc_df["source"] = "email"

print(len(aeslc_df))
aeslc_df.head()

18302


,text,source
0,"Greg/Phillip, Attached is the Grande Communic...",email
1,Phillip & Keith Attached is the first draw re...,email
2,Your Internet Banking accounts are now setup a...,email
3,To our IBS Customers that are still hanging in...,email
4,Phillip Good Morning!\nI hope you had a wonder...,email


In [ ]:
def clean_df(df):

    df = df.copy()

    df["text"] = (
        df["text"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    df = df[
        df["text"] != ""
    ].reset_index(drop=True)

    return df


cnn_df = clean_df(cnn_df)
xsum_df = clean_df(xsum_df)
ag_df = clean_df(ag_df)
aeslc_df = clean_df(aeslc_df)

In [ ]:
SEED = 42

cnn_sample = cnn_df.sample(
    n=125,
    random_state=SEED
)

xsum_sample = xsum_df.sample(
    n=125,
    random_state=SEED
)

ag_sample = ag_df.sample(
    n=125,
    random_state=SEED
)

email_sample = aeslc_df.sample(
    n=125,
    random_state=SEED
)

In [ ]:
eval_df = pd.concat(
    [
        cnn_sample,
        xsum_sample,
        ag_sample,
        email_sample
    ],
    ignore_index=True
)

eval_df = eval_df.sample(
    frac=1,
    random_state=SEED
).reset_index(drop=True)

eval_df["example_id"] = range(
    len(eval_df)
)

print("Total:", len(eval_df))

eval_df["source"].value_counts()

Total: 500


source
ag_news          125
cnn_dailymail    125
xsum             125
email            125
Name: count, dtype: int64

In [ ]:
eval_df[
    ["example_id", "source", "text"]
].head(10)

,example_id,source,text
0,0,ag_news,Powell: N. Korea Blast Not Nuclear Event The U...
1,1,cnn_dailymail,Jerusalem (CNN) -- Two attacks carried out aga...
2,2,ag_news,Former Kan. Junior College Coach Indicted (AP)...
3,3,xsum,The Ethiopian Airlines flight was travelling f...
4,4,cnn_dailymail,"Minami Sanriku, Japan (CNN) -- A 60-year-old ..."
5,5,email,Hi I've attached a couple of files summarizin...
6,6,email,"Hi Kent, I was wondering if we could schedule..."
7,7,cnn_dailymail,Most parents dream of their children growing u...
8,8,cnn_dailymail,"For soccer's world governing body FIFA, it is ..."
9,9,email,"""It's a Girl"" performed by Weave Dance Company..."


In [ ]:
eval_df.to_csv("output.csv", index=False)

In [ ]:
from google.colab import userdata
from groq import Groq

GROQ_API_KEY = "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"

client = Groq(
    api_key=GROQ_API_KEY
)

print("Groq ready")

Groq ready


In [ ]:
models = client.models.list()

available_models = [
    model.id
    for model in models.data
]

for model in available_models:
    print(model)

meta-llama/llama-prompt-guard-2-22m
qwen/qwen3.8-27b
openai/gpt-oss-120b
openai/gpt-oss-safeguard-20b
whisper-large-v3-turbo
qwen/qwen3.6-27b
canopylabs/orpheus-v1-english
canopylabs/orpheus-arabic-saudi
whisper-large-v3
groq/compound
openai/gpt-oss-20b
allam-2-7b
meta-llama/llama-prompt-guard-2-86m
groq/compound-mini


In [ ]:
preferred_models = [
    "openai/gpt-oss-120b",
    "openai/gpt-oss-20b",
    "llama-3.3-70b-versatile"
]

TEACHER_MODEL = next(
    (
        model
        for model in preferred_models
        if model in available_models
    ),
    None
)

if TEACHER_MODEL is None:
    raise ValueError(
        "No preferred teacher available. "
        "Choose one manually from the list above."
    )

print("Teacher:", TEACHER_MODEL)

Teacher: openai/gpt-oss-120b


In [ ]:
SYSTEM_PROMPT = """
Convert the provided English text into concise bullet points containing all materially important information.

Rules:

- Extract all important and independently useful points.
- The number of bullets must depend on the information in the text.
- Do not use a fixed number of bullets.
- Use one bullet per distinct important point.
- Combine details that naturally belong together.
- Remove repetition, filler, and trivial information.
- Preserve important names, dates, numbers, quantities, comparisons, causes, conditions, decisions, and conclusions.
- Do not add information that is not supported by the input.
- Keep every bullet concise while preserving the original meaning.
- Return only bullet points.
- Start every bullet with "- ".
"""

In [ ]:
def teacher_generate(text):

    response = client.chat.completions.create(
        model=TEACHER_MODEL,

        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": text
            }
        ],

        temperature=0,

        max_tokens=1500
    )

    return response.choices[0].message.content.strip()

In [ ]:
text = eval_df.iloc[0]["text"]

print(text)

print("\n--- BULLETS ---\n")

print(
    teacher_generate(text)
)

Powell: N. Korea Blast Not Nuclear Event The United States does not believe that a large explosion in North Korea was related to the communist country #39;s suspected nuclear weapons program, President Bush #39;s foreign policy advisers said Sunday.

--- BULLETS ---

- Powell says the large North Korean explosion was not a nuclear event.  
- U.S. officials do not believe the blast was linked to North Korea’s suspected nuclear weapons program.  
- The statement was made by President Bush’s foreign policy advisers on Sunday.


In [ ]:
from groq import RateLimitError

OUTPUT_CSV = "/content/bullet_benchmark_500_with_references.csv"

# Load already completed IDs if CSV exists
if os.path.exists(OUTPUT_CSV):
    completed_df = pd.read_csv(OUTPUT_CSV)
    completed_ids = set(completed_df["example_id"].astype(int))
    print(f"Already completed: {len(completed_ids)}")
else:
    completed_ids = set()


for _, row in tqdm(
    eval_df.iterrows(),
    total=len(eval_df)
):

    example_id = int(row["example_id"])

    # Skip already completed examples
    if example_id in completed_ids:
        continue

    while True:

        try:

            result = teacher_generate(
                row["text"]
            )

            # Create one-row dataframe
            result_row = pd.DataFrame([{
                "example_id": example_id,
                "source": row["source"],
                "text": row["text"],
                "reference": result
            }])

            # Append immediately to CSV
            result_row.to_csv(
                OUTPUT_CSV,
                mode="a",
                header=not os.path.exists(OUTPUT_CSV),
                index=False
            )

            completed_ids.add(example_id)

            print(
                f"Saved {len(completed_ids)}/{len(eval_df)}"
            )

            time.sleep(20)

            break

        except RateLimitError:

            print(
                f"Rate limited at example {example_id}. "
                "Waiting 120 sec..."
            )

            time.sleep(120)


print("Done:", len(completed_ids))
print("Saved:", OUTPUT_CSV)

Already completed: 6


  0%|          | 0/500 [00:00<?, ?it/s]

Rate limited at example 6. Waiting 60 sec...
Rate limited at example 6. Waiting 60 sec...
Rate limited at example 6. Waiting 60 sec...


KeyboardInterrupt: 

In [ ]:
eval_df["reference"] = references

eval_df[
    [
        "example_id",
        "source",
        "text",
        "reference"
    ]
].head()

In [ ]:
DATASET_FILE = (
    "/content/bullet_benchmark_500.jsonl"
)

eval_df.to_json(
    DATASET_FILE,
    orient="records",
    lines=True,
    force_ascii=False
)

print(DATASET_FILE)

## Models list

In [ ]:
eval_df = pd.read_csv("/content/output_with_bullet_points.csv")

print(eval_df.shape)
eval_df.head()

(500, 4)


,text,source,example_id,bullet_points
0,Powell: N. Korea Blast Not Nuclear Event The U...,ag_news,0,- Powell says the large North Korean explosion...
1,Jerusalem (CNN) -- Two attacks carried out aga...,cnn_dailymail,1,- Two recent attacks on Palestinians sparked I...
2,Former Kan. Junior College Coach Indicted (AP)...,ag_news,2,- Former Kansas junior college basketball coac...
3,The Ethiopian Airlines flight was travelling f...,xsum,3,- Ethiopian Airlines Flight ET500 was travelin...
4,"Minami Sanriku, Japan (CNN) -- A 60-year-old ...",cnn_dailymail,4,"- A 60‑year‑old man, Hiromitsu Shinkawa, was r..."


In [ ]:
MODELS = {

    "smollm_135m":
        "HuggingFaceTB/SmolLM2-135M-Instruct",

    "smollm_360m":
        "HuggingFaceTB/SmolLM2-360M-Instruct",

    "qwen3_06b":
        "Qwen/Qwen3-0.6B",

    "bitnet_2b":
        "microsoft/bitnet-b1.58-2B-4T"
}

MODELS

{'smollm_135m': 'HuggingFaceTB/SmolLM2-135M-Instruct',
 'smollm_360m': 'HuggingFaceTB/SmolLM2-360M-Instruct',
 'qwen3_06b': 'Qwen/Qwen3-0.6B',
 'bitnet_2b': 'microsoft/bitnet-b1.58-2B-4T'}

In [ ]:
DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Using:", DEVICE)

Using: cuda


In [ ]:
def load_model(model_id):

    print(
        "Loading:",
        model_id
    )

    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        trust_remote_code=True
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        trust_remote_code=True,
        torch_dtype="auto",
        device_map="auto"
    )

    model.eval()

    return tokenizer, model

In [ ]:
def build_messages(text):

    return [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": text
        }
    ]

In [ ]:
def generate_with_model(
    tokenizer,
    model,
    text,
    max_new_tokens=512
):

    messages = build_messages(
        text
    )


    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    )


    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }


    input_length = (
        inputs["input_ids"].shape[-1]
    )


    start = time.perf_counter()


    with torch.inference_mode():

        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )


    elapsed = (
        time.perf_counter()
        - start
    )


    generated_tokens = output[
        0,
        input_length:
    ]


    text_output = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()


    output_token_count = len(
        generated_tokens
    )


    tokens_per_second = (
        output_token_count / elapsed
        if elapsed > 0
        else 0
    )


    return {
        "output": text_output,
        "latency_seconds": elapsed,
        "output_tokens": output_token_count,
        "tokens_per_second": tokens_per_second
    }

In [ ]:
def tokenize_messages(
    tokenizer,
    model_id,
    text
):

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": text
        }
    ]


    kwargs = {
        "add_generation_prompt": True,
        "tokenize": True,
        "return_dict": True,
        "return_tensors": "pt"
    }


    if "Qwen3" in model_id:
        kwargs["enable_thinking"] = False


    return tokenizer.apply_chat_template(
        messages,
        **kwargs
    )

In [ ]:
def generate_with_model(
    tokenizer,
    model,
    model_id,
    text,
    max_new_tokens=512
):

    inputs = tokenize_messages(
        tokenizer,
        model_id,
        text
    )


    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }


    input_length = (
        inputs["input_ids"].shape[-1]
    )


    if torch.cuda.is_available():
        torch.cuda.synchronize()


    start = time.perf_counter()


    with torch.inference_mode():

        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )


    if torch.cuda.is_available():
        torch.cuda.synchronize()


    elapsed = (
        time.perf_counter()
        - start
    )


    generated = output[
        0,
        input_length:
    ]


    text_output = tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()


    token_count = len(
        generated
    )


    return {

        "output":
            text_output,

        "latency_seconds":
            elapsed,

        "output_tokens":
            token_count,

        "tokens_per_second":
            (
                token_count / elapsed
                if elapsed > 0
                else 0
            )
    }

In [ ]:
model_name = "smollm_135m"
model_id = MODELS[model_name]

tokenizer, model = load_model(
    model_id
)

example_text = eval_df.iloc[0]["text"]

result = generate_with_model(
    tokenizer,
    model,
    model_id,
    example_text
)

print(result["output"])

print(
    "\nLatency:",
    result["latency_seconds"]
)

print(
    "Tokens/sec:",
    result["tokens_per_second"]
)

Loading: HuggingFaceTB/SmolLM2-135M-Instruct


config.json:   0%|          | 0.00/861 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  269MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Powell: N. Korea Blast Not Nuclear Event The United States does not believe that a large explosion in North Korea was related to the communist country #39;s suspected nuclear weapons program, President Bush #39;s foreign policy advisers said Sunday.

Latency: 11.843042572999991
Tokens/sec: 4.559639101788783


In [ ]:
del model
del tokenizer

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Model unloaded")

Model unloaded


In [ ]:
def benchmark_model(
    model_name,
    model_id,
    dataframe
):

    tokenizer, model = load_model(
        model_id
    )


    rows = []


    for _, row in tqdm(
        dataframe.iterrows(),
        total=len(dataframe),
        desc=model_name
    ):

        try:

            result = generate_with_model(
                tokenizer,
                model,
                model_id,
                row["text"]
            )


            rows.append({

                "example_id":
                    row["example_id"],

                "source":
                    row["source"],

                "model":
                    model_name,

                "output":
                    result["output"],

                "latency_seconds":
                    result["latency_seconds"],

                "output_tokens":
                    result["output_tokens"],

                "tokens_per_second":
                    result["tokens_per_second"]
            })


        except Exception as e:

            rows.append({

                "example_id":
                    row["example_id"],

                "source":
                    row["source"],

                "model":
                    model_name,

                "output":
                    "",

                "latency_seconds":
                    None,

                "output_tokens":
                    None,

                "tokens_per_second":
                    None,

                "error":
                    str(e)
            })


    result_df = pd.DataFrame(
        rows
    )


    output_file = (
        f"/content/results_{model_name}.jsonl"
    )


    result_df.to_json(
        output_file,
        orient="records",
        lines=True,
        force_ascii=False
    )


    del model
    del tokenizer

    gc.collect()


    if torch.cuda.is_available():
        torch.cuda.empty_cache()


    print(
        "Saved:",
        output_file
    )


    return result_df

In [ ]:
smol135_results = benchmark_model(
    "smollm_135m",
    MODELS["smollm_135m"],
    eval_df
)

Loading: HuggingFaceTB/SmolLM2-135M-Instruct


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

smollm_135m:   0%|          | 0/500 [00:00<?, ?it/s]

Saved: /content/results_smollm_135m.jsonl


In [ ]:
smol360_results = benchmark_model(
    "smollm_360m",
    MODELS["smollm_360m"],
    eval_df
)

Loading: HuggingFaceTB/SmolLM2-360M-Instruct


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

smollm_360m:   0%|          | 0/500 [00:00<?, ?it/s]

Saved: /content/results_smollm_360m.jsonl


In [ ]:
qwen_results = benchmark_model(
    "qwen3_06b",
    MODELS["qwen3_06b"],
    eval_df
)

Loading: Qwen/Qwen3-0.6B


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

qwen3_06b:   0%|          | 0/500 [00:00<?, ?it/s]

KeyboardInterrupt: 

## BitNet Model Setup


In [ ]:
!pip uninstall -y transformers
!pip install -q git+https://github.com/huggingface/transformers.git@096f25ae1f501a084d8ff2dcaf25fbc2bd60eba4

Found existing installation: transformers 5.16.1
Uninstalling transformers-5.16.1:
  Successfully uninstalled transformers-5.16.1
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 108.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [ ]:
import transformers

print(transformers.__version__)

from transformers import BitNetForCausalLM

print("BitNet support OK")

4.52.0.dev0
BitNet support OK


In [ ]:
def load_model(model_id):

    print("Loading:", model_id)

    tokenizer = AutoTokenizer.from_pretrained(
        model_id
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map="auto"
    )

    model.eval()

    return tokenizer, model

In [ ]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

MODEL_ID = "microsoft/bitnet-b1.58-2B-4T"

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))


tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID
)


model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID
)


model = model.to("cuda")

model.eval()


print("BitNet loaded successfully")
print("Device:", next(model.parameters()).device)

CUDA available: True
GPU: Tesla T4


You have loaded a BitNet model on CPU and have a CUDA device available, make sure to set your model on a GPU device in order to run your model.


generation_config.json:   0%|          | 0.00/199 [00:00<?, ?B/s]

BitNet loaded successfully
Device: cuda:0


In [ ]:
messages = [
    {
        "role": "system",
        "content": SYSTEM_PROMPT
    },
    {
        "role": "user",
        "content": eval_df.iloc[0]["text"]
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)

input_length = inputs["input_ids"].shape[-1]

with torch.inference_mode():

    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=False
    )

result = tokenizer.decode(
    outputs[0][input_length:],
    skip_special_tokens=True
)

print(result)

/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:641: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


- The United States does not believe that a large explosion in North Korea was related to the communist country's suspected nuclear weapons program.
- President Bush's foreign policy advisers stated this on Sunday.


In [ ]:
def benchmark_bitnet(
    model_name,
    model_id,
    dataframe
):
    import gc
    import time
    import torch
    import pandas as pd

    from tqdm.auto import tqdm
    from transformers import (
        AutoTokenizer,
        AutoModelForCausalLM
    )

    print("Loading:", model_id)

    # -----------------------------------
    # Load WITHOUT device_map="auto"
    # -----------------------------------

    tokenizer = AutoTokenizer.from_pretrained(
        model_id
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_id
    )

    # Move manually instead of Accelerate
    device = (
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    model = model.to(device)
    model.eval()

    print("Loaded successfully")
    print("Device:", device)

    rows = []

    # -----------------------------------
    # Benchmark
    # -----------------------------------

    for _, row in tqdm(
        dataframe.iterrows(),
        total=len(dataframe),
        desc=model_name
    ):

        try:

            messages = [
                {
                    "role": "system",
                    "content": SYSTEM_PROMPT
                },
                {
                    "role": "user",
                    "content": row["text"]
                }
            ]

            inputs = tokenizer.apply_chat_template(
                messages,
                add_generation_prompt=True,
                tokenize=True,
                return_dict=True,
                return_tensors="pt"
            )

            inputs = {
                key: value.to(device)
                for key, value in inputs.items()
            }

            input_length = (
                inputs["input_ids"].shape[-1]
            )

            if device == "cuda":
                torch.cuda.synchronize()

            start = time.perf_counter()

            with torch.inference_mode():

                outputs = model.generate(
                    **inputs,
                    max_new_tokens=512,
                    do_sample=False,
                    pad_token_id=tokenizer.eos_token_id
                )

            if device == "cuda":
                torch.cuda.synchronize()

            elapsed = (
                time.perf_counter()
                - start
            )

            generated_tokens = outputs[
                0,
                input_length:
            ]

            output_text = tokenizer.decode(
                generated_tokens,
                skip_special_tokens=True
            ).strip()

            output_token_count = len(
                generated_tokens
            )

            tokens_per_second = (
                output_token_count / elapsed
                if elapsed > 0
                else 0
            )

            rows.append({
                "example_id": row["example_id"],
                "source": row["source"],
                "model": model_name,
                "output": output_text,
                "latency_seconds": elapsed,
                "output_tokens": output_token_count,
                "tokens_per_second": tokens_per_second
            })

        except Exception as e:

            print(
                f"\nError on example "
                f"{row['example_id']}: {e}"
            )

            rows.append({
                "example_id": row["example_id"],
                "source": row["source"],
                "model": model_name,
                "output": "",
                "latency_seconds": None,
                "output_tokens": None,
                "tokens_per_second": None,
                "error": str(e)
            })

    # -----------------------------------
    # Save
    # -----------------------------------

    result_df = pd.DataFrame(
        rows
    )

    output_file = (
        f"/content/results_{model_name}.jsonl"
    )

    result_df.to_json(
        output_file,
        orient="records",
        lines=True,
        force_ascii=False
    )

    # -----------------------------------
    # Cleanup
    # -----------------------------------

    del model
    del tokenizer

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print()
    print("Saved:", output_file)

    return result_df

In [ ]:
bitnet_results = benchmark_bitnet(
    "bitnet_2b",
    MODELS["bitnet_2b"],
    eval_df
)

Loading: microsoft/bitnet-b1.58-2B-4T


## Evaluation

In [ ]:
smol135_results = pd.read_json(
    "/content/results_smollm_135m.jsonl",
    lines=True)
smol360_results = pd.read_json(
    "/content/results_smollm_360m.jsonl",
    lines=True)
qwen_results = pd.read_json(
    "/content/results_qwen3_06b.jsonl",
    lines=True)
bitnet_results = pd.read_json(
    "/content/bitnet_results.jsonl",
    lines=True)

In [ ]:
bitnet_results["model"] = "bitnet"

In [ ]:
all_results = pd.concat(
    [
        smol135_results,
        smol360_results,
        qwen_results,
        bitnet_results
    ],
    ignore_index=True
)

print(
    all_results.shape
)

all_results.head()

(2000, 9)


,example_id,source,model,output,latency_seconds,output_tokens,tokens_per_second,text,reference
0,0,ag_news,smollm_135m,Powell: N. Korea Blast Not Nuclear Event The U...,1.913454,54,28.221219,NaN,NaN
1,1,cnn_dailymail,smollm_135m,Jerusalem (CNN) - Two attacks on Palestinians ...,19.989540,512,25.613395,NaN,NaN
2,2,ag_news,smollm_135m,"The information provided is: ""Former Kan. Juni...",1.793380,50,27.880318,NaN,NaN
3,3,xsum,smollm_135m,The Ethiopian Airlines flight was traveling fr...,1.250797,35,27.982159,NaN,NaN
4,4,cnn_dailymail,smollm_135m,"Hiromitsu Shinkawa, a 60-year-old Japanese man...",20.429310,512,25.062031,NaN,NaN


In [ ]:
all_results["model"].value_counts(dropna=False)

model
smollm_135m    500
smollm_360m    500
qwen3_06b      500
bitnet         500
Name: count, dtype: int64

In [ ]:
speed_summary = (
    all_results
    .groupby("model")
    .agg(
        avg_latency_seconds=(
            "latency_seconds",
            "mean"
        ),

        median_latency_seconds=(
            "latency_seconds",
            "median"
        ),

        avg_tokens_per_second=(
            "tokens_per_second",
            "mean"
        )
    )
    .reset_index()
)

speed_summary

,model,avg_latency_seconds,median_latency_seconds,avg_tokens_per_second
0,bitnet,15.523463,12.326768,10.981215
1,qwen3_06b,7.311599,6.110800,21.769638
2,smollm_135m,9.909490,8.004976,26.754587
3,smollm_360m,7.088237,5.213456,24.787395


In [ ]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(
    [
        "rouge1",
        "rouge2",
        "rougeL"
    ],
    use_stemmer=True
)

In [ ]:
reference_map = (
    eval_df
    .set_index("example_id")["bullet_points"]
    .to_dict()
)

all_results["reference"] = (
    all_results["example_id"]
    .map(reference_map)
)

In [ ]:
all_results.head()

,example_id,source,model,output,latency_seconds,output_tokens,tokens_per_second,text,reference
0,0,ag_news,smollm_135m,Powell: N. Korea Blast Not Nuclear Event The U...,1.913454,54,28.221219,NaN,- Powell says the large North Korean explosion...
1,1,cnn_dailymail,smollm_135m,Jerusalem (CNN) - Two attacks on Palestinians ...,19.989540,512,25.613395,NaN,- Two recent attacks on Palestinians sparked I...
2,2,ag_news,smollm_135m,"The information provided is: ""Former Kan. Juni...",1.793380,50,27.880318,NaN,- Former Kansas junior college basketball coac...
3,3,xsum,smollm_135m,The Ethiopian Airlines flight was traveling fr...,1.250797,35,27.982159,NaN,- Ethiopian Airlines Flight ET500 was travelin...
4,4,cnn_dailymail,smollm_135m,"Hiromitsu Shinkawa, a 60-year-old Japanese man...",20.429310,512,25.062031,NaN,"- A 60‑year‑old man, Hiromitsu Shinkawa, was r..."


In [ ]:
def calculate_rouge(row):

    if not row["output"]:
        return pd.Series(
            [0, 0, 0]
        )


    scores = scorer.score(
        row["reference"],
        row["output"]
    )


    return pd.Series([
        scores["rouge1"].fmeasure,
        scores["rouge2"].fmeasure,
        scores["rougeL"].fmeasure
    ])


all_results[
    [
        "rouge1",
        "rouge2",
        "rougeL"
    ]
] = all_results.apply(
    calculate_rouge,
    axis=1
)

In [ ]:
quality_summary = (
    all_results
    .groupby("model")
    .agg(
        rouge1=("rouge1", "mean"),
        rouge2=("rouge2", "mean"),
        rougeL=("rougeL", "mean")
    )
    .reset_index()
)

quality_summary

,model,rouge1,rouge2,rougeL
0,bitnet,0.600568,0.404078,0.481380
1,qwen3_06b,0.649384,0.470194,0.546342
2,smollm_135m,0.452839,0.317955,0.372168
3,smollm_360m,0.582672,0.442666,0.489983


In [ ]:
def bullet_format_score(text):

    if not text:
        return 0

    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    if not lines:
        return 0

    correct = sum(
        line.startswith("- ")
        for line in lines
    )

    return correct / len(lines)


all_results["bullet_format_score"] = (
    all_results["output"]
    .apply(bullet_format_score)
)

In [ ]:
format_summary = (
    all_results
    .groupby("model")[
        "bullet_format_score"
    ]
    .mean()
    .reset_index()
)

format_summary

,model,bullet_format_score
0,bitnet,0.999611
1,qwen3_06b,0.999923
2,smollm_135m,0.082755
3,smollm_360m,0.466074


In [ ]:
final_summary = (
    quality_summary
    .merge(
        speed_summary,
        on="model"
    )
    .merge(
        format_summary,
        on="model"
    )
)

final_summary

,model,rouge1,rouge2,rougeL,avg_latency_seconds,median_latency_seconds,avg_tokens_per_second,bullet_format_score
0,bitnet,0.600568,0.404078,0.481380,15.523463,12.326768,10.981215,0.999611
1,qwen3_06b,0.649384,0.470194,0.546342,7.311599,6.110800,21.769638,0.999923
2,smollm_135m,0.452839,0.317955,0.372168,9.909490,8.004976,26.754587,0.082755
3,smollm_360m,0.582672,0.442666,0.489983,7.088237,5.213456,24.787395,0.466074


In [ ]:
domain_summary = (
    all_results
    .groupby(
        ["model", "source"]
    )
    .agg(
        rougeL=(
            "rougeL",
            "mean"
        ),

        latency=(
            "latency_seconds",
            "mean"
        ),

        bullet_format=(
            "bullet_format_score",
            "mean"
        )
    )
    .reset_index()
)

domain_summary

,model,source,rougeL,latency,bullet_format
0,bitnet,ag_news,0.528865,6.067311,1.000000
1,bitnet,cnn_dailymail,0.343146,25.271356,1.000000
2,bitnet,email,0.537578,9.463243,1.000000
3,bitnet,xsum,0.515930,21.291942,0.998442
4,qwen3_06b,ag_news,0.588509,1.822911,1.000000
5,qwen3_06b,cnn_dailymail,0.353340,11.971709,0.999692
6,qwen3_06b,email,0.701712,4.810990,1.000000
7,qwen3_06b,xsum,0.541808,10.640785,1.000000
8,smollm_135m,ag_news,0.481717,4.799274,0.128381
9,smollm_135m,cnn_dailymail,0.262251,13.599538,0.007568
